In [1]:
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models
import torchvision.transforms as T
import scipy.io as sio
import numpy as np

# Usa la funzione get_dataloaders dal tuo script dataset.py
from dataset import get_dataloaders
from customNN import CustomCNN

In [2]:
# 1) Import
from dataset import get_dataloaders

# 2) Percorso al tuo CompCars
root = "../0_dataset/data"

# 3) Trasformazione di test (riduce e normalizza)
import torchvision.transforms as T
test_transform = T.Compose([
    T.Resize((600, 900)),
    T.ToTensor()
])

# 4) Creazione dei DataLoader
#    Nota: passiamo lo stesso train_ratio sia per train che per test
train_loader, test_loader = get_dataloaders(
    root=root,
    train_ratio=0.8,
    batch_size=4,
    seed=123,
    transform=test_transform,
    num_workers=0
)

# 5) Prova a prelevare un batch dal train_loader
batch = next(iter(train_loader))
images     = batch['image']       # [B,3,600,900]
bboxes     = batch['bbox']        # [B,4]
viewpoints = batch['viewpoint']   # [B]
model_ids  = batch['model_id']    # [B]

# 6) Stampa le shapes e qualche valore di esempio
print("=== Train batch ===")
print(" images.shape   →", images.shape)
print(" bboxes.shape   →", bboxes.shape)
print(" viewpoints     →", viewpoints.tolist())
print(" model_ids      →", model_ids.tolist())

# 7) Ripeti sul test_loader
batch_test = next(iter(test_loader))
print("\n=== Test batch shapes ===")
print(" images.shape →", batch_test['image'].shape)
print(" bboxes.shape →", batch_test['bbox'].shape)


TRAIN
Established train dataset paths as:
Dataset root directory: ../0_dataset/data
Image directory: ../0_dataset/data/image
Label directory: ../0_dataset/data/label
Attributes file: ../0_dataset/data/misc/attributes.txt
Mathlab make and models file: ../0_dataset/data/misc/make_model_name.mat

Starting train images extraction... Done

Total number of image extracted: 136726
Starting train images shuffling and selection of 80.0%... Done

Total number if images selected: 109380
Starting attributes loading... Done

Loaded attributes for 1716 models
Starting make and models loading... Done

Loaded 163 makes names
Loaded 2004 models names
=== Train batch ===
 images.shape   → torch.Size([4, 3, 600, 900])
 bboxes.shape   → torch.Size([4, 4])
 viewpoints     → [4, 4, 3, 5]
 model_ids      → [1028, 1941, 1140, 1407]

=== Test batch shapes ===
 images.shape → torch.Size([4, 3, 600, 900])
 bboxes.shape → torch.Size([4, 4])


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for batch in loader:
        inputs = batch['image'].to(device)
        labels = batch['model_id'].to(device) - 1

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += inputs.size(0)

    return running_loss / total, correct / total

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            inputs = batch['image'].to(device)
            labels = batch['model_id'].to(device) - 1
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += inputs.size(0)

    return running_loss / total, correct / total

In [ ]:
root_dir = '../0_dataset/data'
arch = 'custom'  # 'resnet50', 'inception_v3' o 'custom'
epochs = 20
batch_size = 32
train_ratio = 0.8
seed = 42
num_workers = 4

# Use seed for reproducibility
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Extract the number of classes/makes from the dataset
mat_path = os.path.join(root_dir, 'misc', 'make_model_name.mat')
mat = sio.loadmat(mat_path)
num_classes = int(mat.get('model_names').squeeze().shape[0])
print(f"# classes/makes in dataset: {num_classes}")

# Define the transform for the data
input_size = 224
transform_pipeline = T.Compose([
    T.Resize((input_size, input_size)),
    T.ToTensor(),
    #TODO rivedere
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

# Genera DataLoader passando il transform
train_loader, test_loader = get_dataloaders(
    root_dir,
    train_ratio=train_ratio,
    batch_size=batch_size,
    seed=seed,
    transform=transform_pipeline,
    num_workers=num_workers
)

In [28]:
model = CustomCNN(num_classes=num_classes, input_size=input_size)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
best_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
for epoch in range(1, epochs + 1):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    v_loss, v_acc = evaluate(model, test_loader, criterion, device)
    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_loss'].append(v_loss)
    history['val_acc'].append(v_acc)
    print(f"Epoca {epoch}/{epochs} | train_acc: {t_acc:.4f} | val_acc: {v_acc:.4f}")
    if v_acc > best_acc:
        best_acc = v_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"Salvato modello migliore con val_acc: {best_acc:.4f}")
print("Training completato.")

# ### Cell 6: Visualizzazione dei risultati
import matplotlib.pyplot as plt
plt.figure()
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.title('Accuracy per epoca')
plt.xlabel('Epoca')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

Epoca 1/20 | train_acc: 0.0020 | val_acc: 0.0025
Salvato modello migliore con val_acc: 0.0025


KeyboardInterrupt: 